In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, _, _, _ = DATA_DIR_3_x
# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
sales_and_menu_data = load_all_res_3_2_con()

def clean_plain_english_series(series):
    series = (series
              .apply(lambda x: unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('ASCII') if isinstance(x, str) else x) # Step 1: Remove accents/emojis/etc.
              .str.replace(r'[^a-zA-Z0-9\s.,!?;:\'"()\[\]{}\-]', '', regex=True) # Step 2: Remove all non-English characters except basic punctuation
              .str.replace(r'([a-zA-Z])\1{2,}', r'\1\1', regex=True) # Step 3: Limit repeated letters to max 2 (e.g., soooo → soo)
              .str.replace(r'([.,!?;:\'"()\[\]{}\-])\1{1,}', r'\1', regex=True) # Step 4: Limit repeated punctuation to 1 (e.g., !!! → !)
              .str.replace(r'\s+', ' ', regex=True) # Step 5: Normalize whitespace
              .str.strip()
              ) 
    return series

In [ ]:
# Initialize dict all data
sales_menu_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # VLZX7K2M9QD4T doesn't have customers
    if loc_id == 'VLZX7K2M9QD4T':
        sales_menu_customers_data[loc_id] = df
        continue
    
    # Make sure customers are unique
    customers = (
        customers
        .dropna(subset=['customer_id'])
        .drop_duplicates(subset=['location_id', 'customer_id']))

    # Combine customers
    merged = (
        pd.merge(
            df.reset_index().drop(columns='_merge'), # Keep the index, since merges don't keep it 
            customers, 
            on=['location_id', 'customer_id'], how='left', # indicator=True
            )
        .set_index('created_at', drop=False))
    
    # Combine location info
    double_merged = (
        pd.merge(
            merged,
            locations,
            on='location_id', how='left', #indicator=True
        ))
    
    sales_menu_customers_data[loc_id] = double_merged
    
# for loc_id, df in sales_menu_customers_data.items():
#     if loc_id == 'VLZX7K2M9QD4T':
#         continue
#     display(df.dropna(subset='customer_id')['_merge'].value_counts())
#     display(df.dropna(subset='customer_id').query('_merge == "left_only"').head())

In [ ]:
beverages = [
    "Coffee & Tea",
    "Water",
    "Juice",
    "Sports & Health Drink",
    "Soda",
    "Alcohol",
    "Beer",
    "Wine",
    "Beverages",
    "Dairy Drink"]
total = 0
for loc_id, df in sales_menu_customers_data.items():
    
    df = (df
          .assign(item_modifications = lambda df: clean_plain_english_series(df['item_modifications']))
          .pipe(lambda df: df.query('item_type != "Drink"') if loc_id != 'VLZX7K2M9QD4T' else df)
          .query('~dish_category.isin(@beverages)')
          )
    print(loc_id, df.groupby('item_name')['item_modifications'].nunique().sort_values(ascending=False).head(5))
    print()
    print(df.value_counts(['item_name','item_modifications']).size)
    print()
    total += df.value_counts(['item_name','item_modifications']).size
    sales_menu_customers_data[loc_id] = df
    
    df.to_parquet(DATA_DIR_3_3 / f'{loc_id}.parquet')